# Official NVIDIA StyleGAN2-ADA: executed record

10 November: dataset export and official configuration dry run passed; initial CPU session blocked CUDA.
11 November: fresh GPU session resolved hardware access. Two recorded API compatibility patches were required. Compiled CUDA operations, G/D backward, and augmentation + R1 double-backward passed. No model training or optimizer updates.

This notebook preserves historical cell order and outputs, including the earlier CPU report. It is an execution record, not the fresh-runtime entry point. For a clean sequential setup, use notebook 12 or scripts/verify_stylegan2ada_runtime.py. Saved Kaggle Version 2 includes raw outputs and patch files.


In [2]:
from pathlib import Path
import subprocess, sys, json, platform, importlib.util

UPSTREAM_COMMIT = 'd72cc7d041b42ec8e806021a205ed9349f87c6a4'
work = Path('/kaggle/working/stylegan2ada_phase')
assert not work.exists(), 'Choose a fresh work folder; preserve previous evidence'
work.mkdir()
upstream = work / 'stylegan2-ada-pytorch'
subprocess.run(['git','clone','https://github.com/NVlabs/stylegan2-ada-pytorch.git',str(upstream)], check=True)
subprocess.run(['git','checkout','--detach',UPSTREAM_COMMIT], cwd=upstream, check=True)
actual = subprocess.check_output(['git','rev-parse','HEAD'],cwd=upstream,text=True).strip()
assert actual == UPSTREAM_COMMIT
required = ['torch','click','requests','scipy','psutil','tqdm','PIL','ninja','imageio','imageio_ffmpeg']
missing = [name for name in required if importlib.util.find_spec(name) is None]
assert not missing, f'Install official dependencies in a compatible runtime first: {missing}'
import torch
environment = dict(python=platform.python_version(), torch=torch.__version__,
                   cuda=torch.version.cuda, cuda_available=torch.cuda.is_available(),
                   upstream_commit=actual)
(work/'environment.json').write_text(json.dumps(environment,indent=2))
print(environment)


Cloning into '/kaggle/working/stylegan2ada_phase/stylegan2-ada-pytorch'...


{'python': '3.13.16', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'cuda_available': True, 'upstream_commit': 'd72cc7d041b42ec8e806021a205ed9349f87c6a4'}


HEAD is now at d72cc7d Use --resize-filter=box for scaling FFHQ down to 256x256 to match paper FID scores


## Frozen training-data adapter

This cell contains data-validation/export code only. It defines no neural network.
Whole-image Pillow bilinear RGB resize to 128x128 is explicit; no GAN input scaling is done here. The official loader handles scaling. The existing classifier remains RGB /255. This GAN resize implementation differs from the old TensorFlow pilot and is recorded as part of the new recipe.


In [2]:
"""Export the frozen training subset to the official StyleGAN dataset format.

Data adapter only: no GAN architecture, loss, optimizer or training loop.
"""
import argparse
import hashlib
import io
import json
import zipfile
from pathlib import Path

import pandas as pd
from PIL import Image

FINGERPRINT = '4f3037e5dc1726e5a534e3a6f155df3e0e0be52229b057659e097483fd0800cd'
MAPPING = {'NV': 0, 'BKL': 0, 'DF': 0, 'VASC': 0, 'MEL': 1, 'BCC': 1, 'SCC': 1}


def verified_train(frame):
    required = {'image_id', 'label', 'group', 'split', 'original_label', 'target'}
    assert required <= set(frame.columns), 'Manifest columns missing'
    digest = hashlib.sha256(frame[['image_id', 'label', 'group', 'split']]
                           .sort_values('image_id').to_csv(index=False).encode()).hexdigest()
    assert digest == FINGERPRINT, 'Frozen split differs'
    assert len(frame) == 1632 and frame.image_id.is_unique
    assert frame.group.notna().all() and frame.groupby('group').split.nunique().max() == 1
    assert frame.original_label.map(MAPPING).equals(frame.target), 'Target mapping differs'
    assert frame.target.map({0: 'benign', 1: 'malignant'}).equals(frame.label)
    assert frame.split.value_counts().to_dict() == {'train': 972, 'test': 331, 'val': 329}
    train = frame[frame.split == 'train'].sort_values('image_id').reset_index(drop=True)
    assert train.target.value_counts().to_dict() == {0: 530, 1: 442}
    return train


def export_dataset(manifest, input_root, output):
    train = verified_train(pd.read_csv(manifest))
    assert not output.exists(), 'Refusing to overwrite existing dataset'
    wanted = set(train.image_id)
    copies = {}
    for p in sorted(input_root.rglob('*')):
        if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'} and p.stem.upper() in wanted:
            copies.setdefault(p.stem.upper(), []).append(p)
    assert set(copies) == wanted, 'Training image files missing'
    selected = {}
    for image_id, paths in copies.items():
        if len(paths) > 1:
            assert len({hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}) == 1, image_id
        selected[image_id] = paths[0]
    output.parent.mkdir(parents=True, exist_ok=True)
    partial = output.with_name(output.name + '.partial')
    assert not partial.exists(), 'Incomplete previous export exists; choose a new output path'
    labels, evidence = [], []
    with zipfile.ZipFile(partial, 'x', compression=zipfile.ZIP_STORED) as archive:
        for row in train.itertuples(index=False):
            path = selected[row.image_id]
            with Image.open(path) as im:
                im.load()
                # Explicit recipe: whole-image resize, no crop or augmentation.
                resized = im.convert('RGB').resize((128, 128), Image.Resampling.BILINEAR)
            name = f'images/{row.image_id}.png'
            data = io.BytesIO()
            resized.save(data, format='PNG')
            archive.writestr(name, data.getvalue())
            labels.append([name, int(row.target)])
            evidence.append({'image_id': row.image_id, 'target': int(row.target),
                             'source_sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
        archive.writestr('dataset.json', json.dumps({'labels': labels}))
    with zipfile.ZipFile(partial) as archive:
        assert archive.testzip() is None
        assert len(archive.namelist()) == 973
        assert json.loads(archive.read('dataset.json'))['labels'] == labels
    partial.rename(output)
    report = {'dataset': str(output), 'image_count': 972, 'class_counts': {'0': 530, '1': 442},
              'split_sha256': FINGERPRINT,
              'dataset_sha256': hashlib.sha256(output.read_bytes()).hexdigest(),
              'preprocessing': 'Pillow RGB; whole-image bilinear resize to 128x128; uint8 PNG',
              'note': 'Official GAN loader performs its own numeric scaling; CNN preprocessing unchanged',
              'validation_test_image_content_read': False, 'training_sources': evidence}
    output.with_suffix('.audit.json').write_text(json.dumps(report, indent=2))
    print(f'Exported {len(labels)} training images (530 benign / 442 malignant): {output}')
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--manifest', required=True, type=Path)
    parser.add_argument('--input-root', type=Path, default=Path('/kaggle/input'))
    parser.add_argument('--output', required=True, type=Path)
    args = parser.parse_args()
    export_dataset(args.manifest, args.input_root, args.output)


# CLI entry point omitted in notebook.


In [4]:
manifests = sorted(Path('/kaggle/input').rglob('isic2019_binary_split.csv'))
assert len(manifests) == 1, 'Attach exactly one frozen binary manifest'
data_zip = work/'isic_train128.zip'
audit = export_dataset(manifests[0], Path('/kaggle/input'), data_zip)
assert audit['image_count'] == 972


Exported 972 training images (530 benign / 442 malignant): /kaggle/working/stylegan2ada_phase/isic_train128.zip


## Official configuration dry run

Candidate feasibility budget: 20 kimg (20,000 real-image presentations, not 20,000 unique images or 20,000 steps). One GPU, batch 16, seed 142, conditional model, official auto configuration, ADA geometry pipeline (`bg`), no additional dataset mirroring, no pretrained resume, no expensive FID calculation during this smoke candidate. ADA inside GAN training is distinct from classifier conventional augmentation.

This short budget checks feasibility; it cannot establish convergence or clinical validity. Archive all snapshots/grids rather than selecting visually attractive samples. If feasibility succeeds, specify the full run and downstream synthetic budget before executing it.


In [6]:
command = [sys.executable, str(upstream/'train.py'),
           f'--outdir={work / "runs"}', f'--data={data_zip}',
           '--gpus=1','--cfg=auto','--cond=1','--aug=ada','--augpipe=bg',
           '--mirror=0','--batch=16','--kimg=20','--snap=1',
           '--metrics=none','--seed=142','--workers=2']
(work/'candidate_command.json').write_text(json.dumps(command,indent=2))
result = subprocess.run(command+['--dry-run'], cwd=upstream,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
(work/'dry_run.log').write_text(result.stdout)
print(result.stdout)
assert result.returncode == 0, 'Official configuration dry run failed; resolve before training'



Training options:
{
  "num_gpus": 1,
  "image_snapshot_ticks": 1,
  "network_snapshot_ticks": 1,
  "metrics": [],
  "random_seed": 142,
  "training_set_kwargs": {
    "class_name": "training.dataset.ImageFolderDataset",
    "path": "/kaggle/working/stylegan2ada_phase/isic_train128.zip",
    "use_labels": true,
    "max_size": 972,
    "xflip": false,
    "resolution": 128
  },
  "data_loader_kwargs": {
    "pin_memory": true,
    "num_workers": 2,
    "prefetch_factor": 2
  },
  "G_kwargs": {
    "class_name": "training.networks.Generator",
    "z_dim": 512,
    "w_dim": 512,
    "mapping_kwargs": {
      "num_layers": 2
    },
    "synthesis_kwargs": {
      "channel_base": 16384,
      "channel_max": 512,
      "num_fp16_res": 4,
      "conv_clamp": 256
    }
  },
  "D_kwargs": {
    "class_name": "training.networks.Discriminator",
    "block_kwargs": {},
    "mapping_kwargs": {},
    "epilogue_kwargs": {
      "mbstd_group_size": 4
    },
    "channel_base": 16384,
    "channel_max

## Explicit training launch — disabled

Do not enable until the runtime, official source/license, exported labels and dry-run configuration have been reviewed. Preserve the upstream commit and all resolved settings. No custom architecture changes are permitted in this candidate. The first real run also tests CUDA extension compilation and memory use.


In [ ]:
RUN_FEASIBILITY = False
if RUN_FEASIBILITY:
    assert torch.cuda.is_available(), 'A compatible GPU runtime is required'
    assert not (work/'runs').exists(), 'Use a fresh experiment directory'
    subprocess.run(command, cwd=upstream, check=True)
else:
    print('Preparation only. No GAN training launched.')


## Evaluate before classifier augmentation

Review fixed per-class grids across checkpoints, output diversity, class conditioning and near-copy/memorization against training images. Requested labels are not independent clinical validation. Archive all checkpoints, unfiltered samples, settings, hashes and runtime evidence. Only after a satisfactory review should a fixed synthetic-data candidate be evaluated with the unchanged CNN and seeds 42/43/44. Validation remains real-only; the 331 test images remain reserved for the final frozen comparison.

References: [Official implementation](https://github.com/NVlabs/stylegan2-ada-pytorch), [paper](https://arxiv.org/abs/2006.06676), [project transition protocol](../docs/stylegan2ada-transition.md).


In [7]:
import shutil, json, subprocess
RUN_FEASIBILITY = False
smi = shutil.which('nvidia-smi')
smi_result = subprocess.run([smi], capture_output=True, text=True) if smi else None
report = dict(status='data_and_official_dry_run_passed_gpu_runtime_blocked', training_launched=False, optimizer_updates=0, image_count=audit['image_count'], class_counts=audit['class_counts'], dataset_sha256=audit['dataset_sha256'], split_sha256=audit['split_sha256'], environment=environment, nvidia_smi_available=bool(smi), nvidia_smi_output=(smi_result.stdout or smi_result.stderr) if smi_result else None, dry_run_passed=result.returncode == 0 and 'Dry run; exiting.' in result.stdout, limitations='CUDA execution, kernel compilation and model learning remain untested. UI T4 selection did not provide CUDA in this session.')
assert report['dry_run_passed'] and not (work/'runs').exists()
(work/'preflight_report.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
print('PREPARATION_COMPLETE; TRAINING_DISABLED')
shutil.make_archive('/kaggle/working/stylegan2ada_preflight','zip',work)


{
  "status": "data_and_official_dry_run_passed_gpu_runtime_blocked",
  "training_launched": false,
  "optimizer_updates": 0,
  "image_count": 972,
  "class_counts": {
    "0": 530,
    "1": 442
  },
  "dataset_sha256": "028216f4780719022b0200463c0e411a5cc18b64b4f4460e55c49a227a287f6d",
  "split_sha256": "4f3037e5dc1726e5a534e3a6f155df3e0e0be52229b057659e097483fd0800cd",
  "environment": {
    "python": "3.13.18",
    "torch": "2.11.0+cpu",
    "cuda": null,
    "cuda_available": false,
    "upstream_commit": "d72cc7d041b42ec8e806021a205ed9349f87c6a4"
  },
  "nvidia_smi_available": false,
  "nvidia_smi_output": null,
  "dry_run_passed": true,
  "limitations": "CUDA execution, kernel compilation and model learning remain untested. UI T4 selection did not provide CUDA in this session."
}
PREPARATION_COMPLETE; TRAINING_DISABLED


'/kaggle/working/stylegan2ada_preflight.zip'

In [1]:
from pathlib import Path
import platform, shutil, subprocess, json, torch
RUN_FEASIBILITY = False
check = dict(python=platform.python_version(), torch=torch.__version__, torch_cuda=torch.version.cuda, cuda_available=torch.cuda.is_available(), gpu_count=torch.cuda.device_count(), nvidia_smi=shutil.which('nvidia-smi'), nvcc=shutil.which('nvcc'), training_launched=False)
for name in ['nvidia-smi','nvcc']:
    executable = shutil.which(name)
    if executable:
        args = [executable] + (['--version'] if name == 'nvcc' else ['--query-gpu=name,driver_version,memory.total','--format=csv,noheader'])
        r = subprocess.run(args,capture_output=True,text=True)
        check[name+'_output'] = r.stdout or r.stderr
if torch.cuda.is_available():
    x = torch.arange(16,device='cuda',dtype=torch.float32)
    check['cuda_tensor_test'] = float((x*x).sum().cpu()) == 1240.0
    check['gpu_names'] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
Path('/kaggle/working/gpu_runtime_check.json').write_text(json.dumps(check,indent=2))
print(json.dumps(check,indent=2))
print('GPU_RUNTIME_CHECK_COMPLETE')


{
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "torch_cuda": "13.0",
  "cuda_available": true,
  "gpu_count": 2,
  "nvidia_smi": "/opt/bin/nvidia-smi",
  "nvcc": "/usr/local/cuda/bin/nvcc",
  "training_launched": false,
  "nvidia-smi_output": "Tesla T4, 580.178.04, 15360 MiB\nTesla T4, 580.178.04, 15360 MiB\n",
  "nvcc_output": "nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Wed_Aug_20_01:58:59_PM_PDT_2025\nCuda compilation tools, release 13.0, V13.0.88\nBuild cuda_13.0.r13.0/compiler.36424714_0\n",
  "cuda_tensor_test": true,
  "gpu_names": [
    "Tesla T4",
    "Tesla T4"
  ]
}
GPU_RUNTIME_CHECK_COMPLETE


In [5]:
import os, sys, traceback, warnings
os.environ['MAX_JOBS'] = '2'
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
os.environ['TORCH_EXTENSIONS_DIR'] = '/kaggle/working/stylegan_cuda_extensions'
sys.path.insert(0,str(upstream))
from torch_utils.ops import bias_act, upfirdn2d
from torch_utils import custom_ops
custom_ops.verbosity = 'full'
ops_report = dict(environment=environment, upstream_commit=UPSTREAM_COMMIT, training_launched=False, optimizer_updates=0, tests={})
for name, module in [('bias_act',bias_act),('upfirdn2d',upfirdn2d)]:
    print('COMPILING',name,flush=True)
    try:
        assert module._init(), 'Compiled plugin unavailable; fallback is not a compilation pass'
        x = torch.randn(2,8,16,16,device='cuda',requires_grad=True)
        if name == 'bias_act':
            b = torch.randn(8,device='cuda')
            y = module.bias_act(x,b,act='swish',impl='cuda')
            yr = module.bias_act(x,b,act='swish',impl='ref')
        else:
            f = module.setup_filter([1,3,3,1],device='cuda')
            y = module.upfirdn2d(x,f,padding=2,impl='cuda')
            yr = module.upfirdn2d(x,f,padding=2,impl='ref')
        torch.testing.assert_close(y,yr,rtol=1e-4,atol=1e-5)
        g = torch.autograd.grad(y.square().mean(),x,create_graph=True)[0]
        gr = torch.autograd.grad(yr.square().mean(),x,create_graph=True)[0]
        torch.testing.assert_close(g,gr,rtol=1e-4,atol=1e-5)
        gg = torch.autograd.grad(g.square().sum(),x)[0]
        ggr = torch.autograd.grad(gr.square().sum(),x)[0]
        torch.testing.assert_close(gg,ggr,rtol=2e-4,atol=1e-5)
        assert torch.isfinite(gg).all()
        torch.cuda.synchronize()
        ops_report['tests'][name] = dict(passed=True, compiled_plugin_loaded=module._plugin is not None, max_forward_error=float((y-yr).abs().max().detach().cpu()), forward_backward_second_derivative=True)
    except Exception:
        ops_report['tests'][name] = dict(passed=False,error=traceback.format_exc())
    Path('/kaggle/working/stylegan_cuda_ops_check.json').write_text(json.dumps(ops_report,indent=2))
    print(name,ops_report['tests'][name],flush=True)
ops_report['all_passed'] = all(v['passed'] for v in ops_report['tests'].values())
Path('/kaggle/working/stylegan_cuda_ops_check.json').write_text(json.dumps(ops_report,indent=2))
print('OFFICIAL_CUDA_OPS_CHECK_COMPLETE',ops_report['all_passed'])


COMPILING bias_act
Setting up PyTorch plugin "bias_act_plugin"...
Done setting up PyTorch plugin "bias_act_plugin".
bias_act {'passed': True, 'compiled_plugin_loaded': True, 'max_forward_error': 4.76837158203125e-07, 'forward_backward_second_derivative': True}
COMPILING upfirdn2d
Setting up PyTorch plugin "upfirdn2d_plugin"...
Done setting up PyTorch plugin "upfirdn2d_plugin".
upfirdn2d {'passed': True, 'compiled_plugin_loaded': True, 'max_forward_error': 0.0, 'forward_backward_second_derivative': True}
OFFICIAL_CUDA_OPS_CHECK_COMPLETE True


In [4]:
import hashlib, difflib, importlib
original_report = Path('/kaggle/working/stylegan_cuda_ops_check.json')
shutil.copy2(original_report,original_report.with_name('stylegan_cuda_ops_unpatched.json'))
loader = upstream/'torch_utils/custom_ops.py'
original_loader = loader.read_text()
assert original_loader.count('torch.utils.cpp_extension.load(') == 2
assert original_loader.count('module = importlib.import_module(module_name)') == 1
patched_loader = original_loader.replace('torch.utils.cpp_extension.load(', 'module = torch.utils.cpp_extension.load(').replace('        module = importlib.import_module(module_name)\n','')
loader.write_text(patched_loader)
patch_text = ''.join(difflib.unified_diff(original_loader.splitlines(True),patched_loader.splitlines(True),fromfile='a/torch_utils/custom_ops.py',tofile='b/torch_utils/custom_ops.py'))
Path('/kaggle/working/stylegan_pytorch_loader.patch').write_text(patch_text)
patch_info = dict(upstream_commit=UPSTREAM_COMMIT, file='torch_utils/custom_ops.py', original_sha256=hashlib.sha256(original_loader.encode()).hexdigest(), patched_sha256=hashlib.sha256(patched_loader.encode()).hexdigest(), reason='Use cpp_extension.load returned Python module instead of a second import by name.', architecture_changed=False, cuda_kernel_source_changed=False)
Path('/kaggle/working/stylegan_loader_patch.json').write_text(json.dumps(patch_info,indent=2))
importlib.reload(custom_ops)
importlib.reload(bias_act)
importlib.reload(upfirdn2d)
print(patch_text)
print('LOADER_COMPATIBILITY_PATCH_APPLIED')


--- a/torch_utils/custom_ops.py
+++ b/torch_utils/custom_ops.py
@@ -104,11 +104,10 @@
                     # wait until done and continue.
                     baton.wait()
             digest_sources = [os.path.join(digest_build_dir, os.path.basename(x)) for x in sources]
-            torch.utils.cpp_extension.load(name=module_name, build_directory=build_dir,
+            module = torch.utils.cpp_extension.load(name=module_name, build_directory=build_dir,
                 verbose=verbose_build, sources=digest_sources, **build_kwargs)
         else:
-            torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)
-        module = importlib.import_module(module_name)
+            module = torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)
 
     except:
         if verbosity == 'brief':

LOADER_COMPATIBILITY_PATCH_APPLIED


In [9]:
from training.networks import Generator, Discriminator
from training.augment import AugmentPipe
from torch_utils.ops import conv2d_gradfix, grid_sample_gradfix
conv2d_gradfix.enabled = True
grid_sample_gradfix.enabled = True
network_check = dict(training_launched=False, optimizer_updates=0, real_images_read=False, resolution=128, c_dim=2, tests={})
try:
    torch.manual_seed(1142)
    G = Generator(z_dim=512,c_dim=2,w_dim=512,img_resolution=128,img_channels=3,mapping_kwargs=dict(num_layers=2),synthesis_kwargs=dict(channel_base=16384,channel_max=512)).cuda()
    D = Discriminator(c_dim=2,img_resolution=128,img_channels=3,channel_base=16384,channel_max=512).cuda()
    c = torch.eye(2,device='cuda')
    z = torch.randn(2,512,device='cuda')
    fake = G(z,c,noise_mode='const')
    score = D(fake,c)
    score.mean().backward()
    assert torch.isfinite(fake).all() and all(torch.isfinite(p.grad).all() for p in G.parameters() if p.grad is not None)
    network_check['tests']['generator_discriminator_backward'] = True
    G.zero_grad(set_to_none=True)
    D.zero_grad(set_to_none=True)
    a = AugmentPipe(xflip=1,rotate90=1,xint=1,scale=1,rotate=1,aniso=1,xfrac=1).cuda()
    a.p.copy_(torch.ones_like(a.p))
    x = torch.randn(2,3,128,128,device='cuda',requires_grad=True)
    logits = D(a(x),c)
    with conv2d_gradfix.no_weight_gradients():
        dx = torch.autograd.grad(logits.sum(),x,create_graph=True)[0]
    dx.square().sum((1,2,3)).mean().backward()
    assert torch.isfinite(dx).all() and all(torch.isfinite(p.grad).all() for p in D.parameters() if p.grad is not None)
    network_check['tests']['augmentation_and_r1_double_backward'] = True
    network_check['passed'] = True
except Exception:
    network_check['passed'] = False
    network_check['error'] = traceback.format_exc()
finally:
    Path('/kaggle/working/stylegan_network_check.json').write_text(json.dumps(network_check,indent=2))
    print(json.dumps(network_check,indent=2))
    print('SYNTHETIC_NETWORK_CHECK_COMPLETE')


/kaggle/working/stylegan2ada_phase/stylegan2-ada-pytorch/torch_utils/ops/conv2d_gradfix.py:55: UserWarning: conv2d_gradfix not supported on PyTorch 2.11.0+cu130. Falling back to torch.nn.functional.conv2d().
  warnings.warn(f'conv2d_gradfix not supported on PyTorch {torch.__version__}. Falling back to torch.nn.functional.conv2d().')
/kaggle/working/stylegan2ada_phase/stylegan2-ada-pytorch/torch_utils/ops/conv2d_gradfix.py:55: UserWarning: conv2d_gradfix not supported on PyTorch 2.11.0+cu130. Falling back to torch.nn.functional.conv2d().
  warnings.warn(f'conv2d_gradfix not supported on PyTorch {torch.__version__}. Falling back to torch.nn.functional.conv2d().')
/kaggle/working/stylegan2ada_phase/stylegan2-ada-pytorch/torch_utils/ops/conv2d_gradfix.py:55: UserWarning: conv2d_gradfix not supported on PyTorch 2.11.0+cu130. Falling back to torch.nn.functional.conv2d().
  warnings.warn(f'conv2d_gradfix not supported on PyTorch {torch.__version__}. Falling back to torch.nn.functional.conv2d(

{
  "training_launched": false,
  "optimizer_updates": 0,
  "real_images_read": false,
  "resolution": 128,
  "c_dim": 2,
  "tests": {
    "generator_discriminator_backward": true,
    "augmentation_and_r1_double_backward": true
  },
  "passed": true
}
SYNTHETIC_NETWORK_CHECK_COMPLETE


In [8]:
grid_file = upstream/'torch_utils/ops/grid_sample_gradfix.py'
grid_original = grid_file.read_text()
start = grid_original.index('def _should_use_custom_op():')
end = grid_original.index('#----------------------------------------------------------------------------',start)
grid_patched = grid_original[:start] + 'def _should_use_custom_op():\n    return enabled\n\n' + grid_original[end:]
old_call = "        op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')\n        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False)"
new_call = "        op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')[0]\n        output_mask = (ctx.needs_input_grad[1], ctx.needs_input_grad[2])\n        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False, output_mask)"
assert grid_patched.count(old_call) == 1
grid_patched = grid_patched.replace(old_call,new_call)
grid_file.write_text(grid_patched)
grid_patch = ''.join(difflib.unified_diff(grid_original.splitlines(True),grid_patched.splitlines(True),fromfile='a/torch_utils/ops/grid_sample_gradfix.py',tofile='b/torch_utils/ops/grid_sample_gradfix.py'))
Path('/kaggle/working/stylegan_grid_gradfix.patch').write_text(grid_patch)
shutil.copy2('/kaggle/working/stylegan_network_check.json','/kaggle/working/stylegan_network_before_grid_fix.json')
grid_patch_info = dict(source='https://github.com/NVlabs/stylegan3/blob/main/torch_utils/ops/grid_sample_gradfix.py', changes='Port NVIDIA newer API handling: use enabled gradfix, operation tuple[0], backward output_mask. Target runtime PyTorch 2.11 only.', original_sha256=hashlib.sha256(grid_original.encode()).hexdigest(), patched_sha256=hashlib.sha256(grid_patched.encode()).hexdigest(), architecture_changed=False)
Path('/kaggle/working/stylegan_grid_patch.json').write_text(json.dumps(grid_patch_info,indent=2))
importlib.reload(grid_sample_gradfix)
grid_sample_gradfix.enabled = True
print(grid_patch)
print('NVIDIA_GRID_API_COMPATIBILITY_UPDATE_APPLIED')


--- a/torch_utils/ops/grid_sample_gradfix.py
+++ b/torch_utils/ops/grid_sample_gradfix.py
@@ -32,12 +32,7 @@
 #----------------------------------------------------------------------------
 
 def _should_use_custom_op():
-    if not enabled:
-        return False
-    if any(torch.__version__.startswith(x) for x in ['1.7.', '1.8.', '1.9']):
-        return True
-    warnings.warn(f'grid_sample_gradfix not supported on PyTorch {torch.__version__}. Falling back to torch.nn.functional.grid_sample().')
-    return False
+    return enabled
 
 #----------------------------------------------------------------------------
 
@@ -61,8 +56,9 @@
 class _GridSample2dBackward(torch.autograd.Function):
     @staticmethod
     def forward(ctx, grad_output, input, grid):
-        op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')
-        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False)
+        op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')[0]
+    

In [10]:
import zipfile
base = Path('/kaggle/working')
runtime_report = dict(date='2026-11-11', status='gpu_and_targeted_runtime_checks_passed', hardware=check, cuda_ops=ops_report, network_check=network_check, loader_patch=patch_info, grid_patch=grid_patch_info, conv2d_gradfix_native_fallback=True, training_launched=False, optimizer_updates=0, real_image_content_read=False, limitations='Targeted synthetic checks only; path-length regularization, full training loop, checkpoints, convergence and image quality are not yet validated. Notebook contains historical preparation outputs; use the compatibility script for fresh runtime setup.')
assert check['cuda_tensor_test'] and ops_report['all_passed'] and network_check['passed']
(base/'stylegan_gpu_runtime_report.json').write_text(json.dumps(runtime_report,indent=2))
with zipfile.ZipFile(base/'stylegan_gpu_runtime_evidence.zip','w',compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ['gpu_runtime_check.json','stylegan_cuda_ops_check.json','stylegan_cuda_ops_unpatched.json','stylegan_network_check.json','stylegan_network_before_grid_fix.json','stylegan_pytorch_loader.patch','stylegan_loader_patch.json','stylegan_grid_gradfix.patch','stylegan_grid_patch.json','stylegan_gpu_runtime_report.json']:
        archive.write(base/name,name)
print('GPU RUNTIME: PASS')
print('CUDA compiled operations: PASS; first and second derivatives checked')
print('Generator/discriminator backward: PASS')
print('ADA geometry plus R1 double-backward: PASS')
print('COMPATIBILITY: two recorded API patches; native conv2d fallback')
print('TRAINING: DISABLED; optimizer updates: 0')
print('Saved stylegan_gpu_runtime_evidence.zip')


GPU RUNTIME: PASS
CUDA compiled operations: PASS; first and second derivatives checked
Generator/discriminator backward: PASS
ADA geometry plus R1 double-backward: PASS
COMPATIBILITY: two recorded API patches; native conv2d fallback
TRAINING: DISABLED; optimizer updates: 0
Saved stylegan_gpu_runtime_evidence.zip
